# Günlük sayaç hedefleri: değer, aralık ve belirsizlik

Bu inceleme UTC takvimini kullanır. Sonuçlar kayıtlı kümülatif endeksin sürekliliği varsayımına bağlıdır; fiziksel ölçüm doğruluğu veya doğrulanmış litre değildir. Model eğitimi kapalıdır.

In [ ]:
from pathlib import Path
import json
import pyarrow.parquet as pq
from urban_water.db import PROJECT_ROOT
from urban_water.datasets import file_sha
from urban_water_research.counter_days import verify_counter_day_run

pointer = json.loads((PROJECT_ROOT / 'var/research/hihaigua/v2/counter-days/latest.json').read_text(encoding='utf-8'))
directory = Path(pointer['directory'])
assert file_sha(directory / 'manifest.json') == pointer['manifest_sha256']
verification = verify_counter_day_run(directory)
assert verification['algorithm_matches_current']
result = json.loads((directory / 'manifest.json').read_text(encoding='utf-8'))
{'verification': verification, 'components': {c: s['statuses'] for c, s in result['components'].items()}}


In [ ]:
examples = {}
columns = ['household_code', 'target_date', 'status', 'point_target_source',
           'volume_lower_source', 'volume_upper_source', 'reasons', 'source_unit',
           'label_available_at_sample_clock', 'archive_received_at', 'training_allowed']
for status in ['exact_counter', 'bounded_counter', 'unresolved']:
    table = pq.read_table(directory / 'water-daily.parquet', columns=columns, filters=[('status', '=', status)])
    examples[status] = table.slice(0, 1).to_pylist()
examples


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(directory / 'target-status-by-household.png')))


Tek değer verilemeyen günler için aralık ortalaması alınmaz, eksik günler sıfır yapılmaz. `label_available_at_sample_clock`, gün sonrasındaki gerekli ilk örneği de dikkate alır; tarihsel sağlayıcı geliş zamanı değildir.

Yalnız tek değerli günleri seçmek, gece tüketimi ve hane/veri kapsaması bakımından örneklem seçilimine yol açabilir. Bu tablo bir eğitim veya model başarısı sonucu değildir. Ayrıntılar: `docs/hihaigua-counter-days.md`.